In [7]:
%pip install fxencoder_plusplus
import torch
from fxencoder_plusplus import load_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = load_model('default', device=device)  # auto-downloads weights
print(f'Using device: {device}')

Note: you may need to restart the kernel to use updated packages.
Description: Default model
Model downloaded successfully to: /Users/milanliessens/.cache/fxencoder_plusplus/models--yytung--fxencoder-plusplus/snapshots/d6ed786a5fa20126402121ba8e28e1a6427eea52/fxenc_plusplus_default.pt


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load our best checkpoint in the paper.
The checkpoint is already downloaded
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm2.we

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load our best checkpoint in the paper.
The checkpoint is already downloaded
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm2.we

In [ ]:
import inspect
import fxencoder_plusplus
from fxencoder_plusplus import load_model

print('module:', fxencoder_plusplus.__file__)
print('load_model:', inspect.signature(load_model))

module: /Users/milanliessens/miniconda3/envs/pwfx/lib/python3.10/site-packages/fxencoder_plusplus/__init__.py
load_model: (model_name='default', model_path=None, device='cuda', auto_download=True, cache_dir=None)
get_fx_embedding: (x)
get_fx_embedding_by_audio_query: (x, audio_query)
get_fx_embedding_by_text_query: (x, text_query)


In [16]:
import torch, librosa

In [ ]:
def get_fx_embedding(audio_path):
    """Get the FX embedding for a given audio file."""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    wav, sr = librosa.load(audio_path, sr=44100, mono=False)
    wav = torch.from_numpy(wav).float()

    if wav.ndim == 1:
        wav = wav.unsqueeze(0).repeat(2, 1)
    elif wav.ndim == 2 and wav.shape[0] == 1:
        wav = wav.repeat(2, 1)
    elif wav.ndim != 2:
        raise ValueError(f'Unexpected waveform shape: {tuple(wav.shape)}')

    wav = wav.unsqueeze(0).to(device)  # [1, 2, seq_len]

    return model.get_fx_embedding(wav)


def cosine_similarity_from_audio(audio_path_a, audio_path_b):
    """Compute cosine similarity between FX embeddings of two audio files."""
    emb_a = get_fx_embedding(audio_path_a)
    emb_b = get_fx_embedding(audio_path_b)

    # Flatten to [D] so we can compute scalar cosine similarity.
    emb_a = emb_a.reshape(-1)
    emb_b = emb_b.reshape(-1)

    sim = torch.nn.functional.cosine_similarity(
        emb_a.unsqueeze(0), emb_b.unsqueeze(0), dim=1
    )
    return sim.item()

In [ ]:
import torch, librosa

audio_path = '../data/audio/piano.wav'
get_fx_embedding(audio_path)

tensor([[ 0.0808,  0.0770,  0.0957,  0.1110,  0.0515, -0.1236,  0.0279,  0.1533,
         -0.0370, -0.0651,  0.0311, -0.0588,  0.1364,  0.0924,  0.1113, -0.0980,
         -0.0719,  0.0931, -0.1553,  0.0160, -0.0220,  0.0199,  0.1507, -0.0825,
          0.1064,  0.0483,  0.0181,  0.0219,  0.0484, -0.1445,  0.0021, -0.0553,
          0.1388, -0.0912, -0.0302, -0.1203,  0.0750, -0.1202, -0.0134, -0.0920,
         -0.1867,  0.0293, -0.1533,  0.0409,  0.2034, -0.0101, -0.0924, -0.0266,
         -0.0429, -0.0883, -0.0331,  0.0033, -0.0845,  0.0177, -0.0991, -0.1137,
         -0.0522,  0.1617,  0.1060,  0.0420, -0.1030, -0.1893, -0.0381,  0.0954,
         -0.1062,  0.0814,  0.0244,  0.0200, -0.0185, -0.0105, -0.1105, -0.1445,
         -0.0460, -0.0276,  0.0257, -0.1027, -0.0361,  0.1090, -0.0542, -0.1552,
          0.1276,  0.0730, -0.0642, -0.0270, -0.0788,  0.0126, -0.0491,  0.0377,
          0.0605, -0.1516, -0.0872, -0.0907,  0.0108, -0.1449,  0.0368, -0.0915,
         -0.0877,  0.0285, -

In [21]:
audio_path_a = '../data/audio/piano.wav'
audio_path_b = '../data/audio/piano.wav'

similarity = cosine_similarity_from_audio(audio_path_a, audio_path_b)
print(f'Cosine similarity: {similarity:.4f}')

Cosine similarity: 1.0000


In [ ]:
audio_path_a = '../results/experiment_2026-03-08_23-24-52/piano/LLM+LLM/run_1/intermediate/00_original.wav'
audio_path_b = '../results/experiment_2026-03-08_23-24-52/piano/LLM+LLM/run_1/intermediate/01_initialized.wav'
audio_path_c = '../results/experiment_2026-03-08_23-24-52/piano/LLM+LLM/run_1/intermediate/02_refined.wav'

similarity = cosine_similarity_from_audio(audio_path_a, audio_path_b)
print(f'Cosine similarity: {similarity:.4f}')

Cosine similarity: 0.5319


In [27]:
similarity = cosine_similarity_from_audio(audio_path_c, audio_path_b)
print(f'Cosine similarity: {similarity:.4f}')

Cosine similarity: 0.9571
